<a href="https://colab.research.google.com/github/abdulhaseeb941/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulhaseeb941/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Classification**

My decline_recovery lane is a binary classification problem: for each page, predict
whether it belongs to the "declining" class or not (is_declining_label =
trend_direction == "down"). It's not clustering (I'm not discovering unlabeled groups —
the label already exists), not ranking on its own (ranking is what I do *with* the
classifier's output afterward, to prioritize review), and not pure scoring in the
regression sense (I'm not predicting a continuous number like exact traffic change —
I'm predicting a yes/no class). The notebook 01/02 pipeline already treats it this way:
a binary label, scored with Precision@K after ranking by predicted probability.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, subprocess

REPO_URL = "https://github.com/abdulhaseeb941/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL], check=True)

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

Working dir: /content/flyrank-ml-internship


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: is_declining_label**

I'd predict is_declining_label = (trend_direction == "down"), the same label the
starter pipeline already defines. This is a proxy, not a directly observed business
outcome — trend_direction itself is derived from a traffic/ranking trend calculation
over some prior window, not something a human manually labeled as "this page declined."
That matters: the model is learning to agree with a particular trend-detection rule,
not learning "decline" in some deeper sense. I'd want to sanity-check how trend_direction
itself is computed (window length, threshold for "down") before trusting the label too
far, since any noise or bias baked into that upstream rule becomes noise or bias in my
model's target.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@50**

I'd defend Precision@K (specifically Precision@50, matching notebook 01's evaluation)
as the metric that matters here, not overall accuracy. The real action this model
supports is a content team reviewing a ranked list of pages, starting from the top —
they only have time to look at the first N pages each cycle, not the whole dataset.
Precision@50 directly answers "of the top 50 pages I'd hand the team, how many are
actually declining?" which maps onto their real workflow. Accuracy would be misleading
here anyway, since the baseline shows ~54% of pages are already declining in this
sample — a model could score high accuracy by just guessing the majority class,
without being useful for prioritization at all.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# One row = one content page (content_id), at the time this snapshot was taken.
lane_slice = df[["content_id", "client_id", "trend_direction", "days_since_last_update",
                  "impressions_90d", "avg_position", "ctr", "word_count"]].copy()
lane_slice["is_declining_label"] = (lane_slice["trend_direction"] == "down").astype(int)

print(f"Unit of analysis: one row = one content page. Shape: {lane_slice.shape}")
lane_slice.head(10)


Unit of analysis: one row = one content page. Shape: (30000, 9)


,content_id,client_id,trend_direction,days_since_last_update,impressions_90d,avg_position,ctr,word_count,is_declining_label
0,content_304f48230142,client_f369cb89fc,down,20,3803,10.6,0.76,3221.0,1
1,content_a1fb4e703a9e,client_4e07408562,down,25,15320,20.3,0.05,2481.0,1
2,content_9aa793d4d895,client_7f2253d7e2,down,20,12581,36.5,0.09,3515.0,1
3,content_331d6c4de07b,client_19581e27de,stable,22,11751,6.2,0.49,NaN,0
4,content_d99b7a2d90ca,client_3fdba35f04,down,14,19140,44.0,0.13,2803.0,1
5,content_d4084a4bc775,client_f369cb89fc,down,20,3970,8.5,0.03,3080.0,1
6,content_9a34b442b552,client_8722616204,down,20,20,7.0,0.00,3059.0,1
7,content_a63219c6e95a,client_19581e27de,stable,22,1724,21.2,0.06,NaN,0
8,content_5e6c160719bc,client_6208ef0f77,down,20,32574,46.0,0.09,3807.0,1
9,content_c27558df2b0c,client_19581e27de,down,104,1240,4.9,0.16,NaN,1


The dataframe above shows the unit of analysis directly: one row = one content page
(content_id), identified alongside its client (client_id) and the features used to
predict decline. The is_declining_label column at the end is the target computed from
trend_direction, showing 0/1 values alongside the features that would feed the model.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**A fixed rule already exists here, and it's weak — that's the evidence.**

The starter pipeline's hand-written baseline rule only reaches Precision@50 = 0.240,
while a trained model reaches roughly 0.70-0.75 on the same task — about a 3x
improvement. That gap is direct evidence the pattern is too messy for a simple
if-statement: if one or two features cleanly separated declining pages from stable
ones, a fixed rule would already score close to the model.

I also tested this myself in notebook 01: the single most "obvious" rule-of-thumb
signal, days_since_last_update, barely differs between declining pages (avg 49 days)
and non-declining pages (avg 42 days) — only a 7-day gap, not enough to build a
reliable rule on. Real decline risk depends on a combination of signals (position,
impressions, staleness, word count, etc.) interacting in ways that aren't obviously
additive or threshold-based, which is exactly the kind of non-linear pattern a model
can pick up on and a human-written rule can't easily hand-code.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.